# AlphaZero

AlphaZero was a program that was developed to play general 2-player perfect information games by training a model on self-play.

## Connect 4

We will implement Connect 4 to demonstrate the algorithm. First, we import some libraries for utility.

In [10]:

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

# Select device (CUDA if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


We define a class that will allow us to play Connect 4 games.

In [11]:


class ConnectFour:

    def __init__(self, rows=6, columns=7):

        self.rows = rows
        self.columns = columns
        self.board = torch.zeros((rows, columns), dtype=torch.int)
        self.current_player = 1
        self.is_terminal = False
        self.winner = 0
    
    
    def __str__(self):
        return str(self.board.numpy()) + "\nCurrent Player: " + str(self.current_player) + "\nTerminal: " + str(self.is_terminal)

    def reset(self):
        self.board = torch.zeros((self.rows, self.columns), dtype=torch.int)
        self.current_player = 1
        self.is_terminal = False
        self.winner = 0

    def copy(self):
        new_game = ConnectFour(self.rows, self.columns)
        new_game.board = self.board.clone()
        new_game.current_player = self.current_player
        new_game.is_terminal = self.is_terminal
        new_game.winner = self.winner
        return new_game

    def get_actions(self):
        # If the topmost row is empty, the column is available
        return self.board[0, :] == 0

    def play_action(self, column):
        '''
        Returns 0 if the game continues, 1 for wins, 2 for draws
        '''

        if self.board[0, column] == 0:
            row = self.rows - 1
            while row >= 0 and self.board[row, column] != 0:
                row -= 1

            if row >= 0:
                self.board[row, column] = self.current_player
                if self.check_winner(row, column):
                    self.is_terminal = True
                    self.winner = float(self.current_player)
                    self.current_player *= -1
                    return 1
                self.current_player *= -1
                if self.get_actions().sum() == 0:
                    self.is_terminal = True
                    return 2
            return 0

        return 0

    def check_winner(self, row, column):

        # Check lines at (row, column)
        player = self.board[row, column]
        for dr, dc in [(1, 0), (0, 1), (1, 1), (1, -1)]:
            total = 1
            for i in reversed(range(-3, 0)):
                r = row + i * dr
                c = column + i * dc
                if 0 <= r < self.rows and 0 <= c < self.columns and self.board[r, c] == player:
                    total += 1
                    if total == 4:
                        return player
                else:
                    break

            for i in range(1, 4):
                r = row + i * dr
                c = column + i * dc
                if 0 <= r < self.rows and 0 <= c < self.columns and self.board[r, c] == player:
                    total += 1
                    if total == 4:
                        return player
                else:
                    break
        return 0


    


In [12]:

# test_game = ConnectFour()
# print(test_game)

# while test_game.get_actions().sum() > 0 and not test_game.is_terminal:

#     action = int(input("Enter column (0-6): "))
#     test_game.play_action(action)
#     print(test_game)


## Policy and Value Networks

We will use CNN models for the policy and value network.

In [13]:


class PolicyValueNetwork(nn.Module):

    def __init__(self, input_shape=(1, 6, 7), num_actions=7, device: torch.device = device):
        super().__init__()
        self.device = device
        self.conv1 = nn.Conv2d(input_shape[0], 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.fc_policy = nn.Linear(64 * input_shape[1] * input_shape[2], num_actions)
        self.fc_value = nn.Linear(64 * input_shape[1] * input_shape[2], 1)
        self.to(self.device)

    def forward(self, x, softmax=True):
        x = x.to(self.device)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(x.size(0), -1)  # Flatten
        policy_logits = self.fc_policy(x)
        value = torch.tanh(self.fc_value(x))
        return (F.softmax(policy_logits, dim=-1) if softmax else policy_logits), value


## Monte Carlo Tree Search (MCTS)

We implement MCTS with PUCT (Predictor + Upper Confidence Bound for Trees) to select nodes.

In [14]:

class AlphaZero:
    class MCTSNode:
        def __init__(self, state, prior=0.0, parent=None):
            self.state = state
            self.prior = float(prior)          # P(s,a)
            self.parent = parent               # parent node or None for root
            self.visits = 0                    # N(s,a)
            self.value_sum = 0.0               # W(s,a)
            self.children = {}                 # action -> child node

        def q_value(self) -> float:
            # Average value; 0 if unvisited
            return self.value_sum / self.visits if self.visits > 0 else 0.0

        def u_value(self, c_puct: float = 1.5) -> float:
            # Exploration bonus using parent visits; safe for root/unvisited
            parent_visits = self.parent.visits if (self.parent and self.parent.visits > 0) else 1
            return c_puct * self.prior * np.sqrt(parent_visits) / (1.0 + self.visits)

        def puct(self, c_puct: float = 1.5) -> float:
            return self.q_value() + self.u_value(c_puct)

        def select_child(self, c_puct: float = 1.5):
            # Choose the child with max PUCT score
            return max(self.children.values(), key=lambda ch: ch.puct(c_puct))

        def expand(self, priors):
            # priors: 1D numpy array of length num_actions (already masked/normalized)
            valid = self.state.get_actions().numpy()
            for action, prior in enumerate(priors):
                if valid[action] and action not in self.children:
                    # Create a new child node for the action
                    child_state = self.state.copy()
                    child_state.play_action(action)
                    self.children[action] = AlphaZero.MCTSNode(child_state, prior, parent=self)
            

    def __init__(self, model=PolicyValueNetwork()):
        self.model = model

    @staticmethod
    def encode_state(state: ConnectFour) -> torch.Tensor:
        # Encode from the perspective of the current player (1 or -1)
        # Shape: (1, 1, rows, cols)
        x = state.board.clone().float().unsqueeze(0).unsqueeze(0)
        x = x * state.current_player
        return x

    @staticmethod
    def mask_and_normalize_priors(priors_t: torch.Tensor, state: ConnectFour) -> np.ndarray:
        # Convert priors to 1D numpy and mask invalid moves, then renormalize
        priors = priors_t.squeeze(0).detach().cpu().numpy()
        valid = state.get_actions().numpy().astype(np.float32)
        priors = priors * valid
        s = priors.sum()
        if s > 0:
            priors = priors / s
        else:
            # If all priors were zero after masking, fall back to uniform over valid moves
            v = valid.sum()
            priors = valid / v if v > 0 else valid
        return priors.astype(np.float32)

    def add_dirichlet_noise(self, node, epsilon=0.25, alpha=0.03):
        if not node.children:
            return
        num_actions = max(node.children.keys()) + 1
        # Build prior vector aligned to action indices
        child_actions = sorted(node.children.keys())
        noise = np.random.dirichlet([alpha] * len(child_actions))
        for i, a in enumerate(child_actions):
            child = node.children[a]
            child.prior = float((1 - epsilon) * child.prior + epsilon * noise[i])

    def MCTS(self, state, num_simulations=200, train=True, temp=1.0, c_puct=1.414):

        root = self.MCTSNode(state)

        # Expand root once and optionally add Dirichlet noise
        with torch.no_grad():
            priors_t, value_t = self.model(self.encode_state(root.state).to(self.model.device))
        
        priors = self.mask_and_normalize_priors(priors_t, root.state)
        root.expand(priors)
        if train:
            # Typical AlphaZero settings: epsilon=0.25, alpha dependent on branching factor
            self.add_dirichlet_noise(root, epsilon=0.25, alpha=0.03)

        for _ in range(num_simulations):
            node = root
            # SELECTION
            while node.children:
                node = node.select_child(c_puct)

            # EVALUATION + EXPANSION (skip expansion if terminal)
            if not node.state.is_terminal:
                with torch.no_grad():
                    priors_t, value_t = self.model(self.encode_state(node.state).to(self.model.device))
                priors = self.mask_and_normalize_priors(priors_t, node.state)
                node.expand(priors)
                v = -float(value_t.squeeze().item())
            else:
                v = -(node.state.winner * node.state.current_player)

            # BACKPROPAGATION (flip sign along the path)
            while node is not None:
                node.visits += 1
                node.value_sum += v
                v = -v
                node = node.parent
            
        # Build action probabilities from visit counts at root
        visits = np.zeros(root.state.columns, dtype=np.float32)
        for action, child in root.children.items():
            visits[action] = child.visits
        if temp <= 1e-6:
            # Greedy
            probs = np.zeros_like(visits)
            best = int(visits.argmax())
            probs[best] = 1.0
        else:
            powered = np.power(visits, 1.0 / temp)
            s = powered.sum()
            probs = powered / s if s > 0 else powered

        if train:
            # Return action probabilities as a numpy array or torch tensor per your pipeline
            return probs
        else:
            # Return best action for inference
            return int(np.argmax(probs))
    


## Training the Network

We run self-play games to train the network to play Connect 4.

In [15]:


def generate_game(alphazero_model):

    history = []
    distributions = []
    values = []
    game = ConnectFour()

    while not game.is_terminal:

        history.append((game.board.clone() * game.current_player).float().unsqueeze(0).unsqueeze(0).to(alphazero_model.model.device))

        distribution = torch.tensor(alphazero_model.MCTS(game), device=alphazero_model.model.device)
        distributions.append(distribution)

        action = torch.multinomial(distribution, num_samples=1).item()
        game.play_action(action)

    value = game.winner
    for _ in range(len(history)):
        values.append(value)
        value *= -1
    
    return history, distributions, values


In [18]:


def train(alphazero_model, batch_size=64, epochs=1024):

    optimizer = optim.Adam(alphazero_model.model.parameters(), lr=0.001)
    policy_criterion = nn.KLDivLoss(reduction='batchmean')  # We'll compare log probs to target dist
    value_criterion = nn.MSELoss()

    for epoch in range(1,epochs + 1):

        states = []
        policy_targets = []
        value_targets = []
        for game in range(batch_size):

            history, distributions, values = generate_game(alphazero_model)
            states += history
            policy_targets += distributions
            value_targets += values
            print(f"Game {game} ended in:", values[0])
        
        states = torch.cat(states, dim=0).to(alphazero_model.model.device)  # shape (N,1,6,7)
        policy_targets = torch.stack(policy_targets).to(alphazero_model.model.device)  # shape (N,7)
        value_targets = torch.tensor(value_targets, dtype=torch.float32, device=alphazero_model.model.device).unsqueeze(1)  # shape (N,1)

        optimizer.zero_grad()

        policy_logits, value_predictions = alphazero_model.model.forward(states, softmax=False)
        log_policy = F.log_softmax(policy_logits, dim=-1)
        
        policy_loss = policy_criterion(log_policy, policy_targets)
        value_loss = value_criterion(value_predictions, value_targets)

        total_loss = policy_loss + value_loss
        total_loss.backward()

        optimizer.step()
        print(f"Epoch {epoch}: Loss:{total_loss.item():.4f} Policy:{policy_loss.item():.4f} Value:{value_loss.item():.4f}")
        


In [ ]:


alphazero = AlphaZero(PolicyValueNetwork(device=device))

# Quick sanity check run (reduce epochs for initial test)
train(alphazero_model=alphazero, batch_size=16, epochs=100)


Game 0 ended in: -1.0
Game 1 ended in: 1.0
Game 2 ended in: -1.0
Game 3 ended in: 1.0
Game 4 ended in: 1.0
Game 5 ended in: 1.0
Game 6 ended in: -1.0
Game 7 ended in: -1.0
Game 8 ended in: -1.0
Game 9 ended in: -1.0
Game 10 ended in: 1.0
Game 11 ended in: 1.0
Game 12 ended in: -1.0
Game 13 ended in: 1.0
Game 14 ended in: 1.0
Game 15 ended in: -1.0
Epoch 1: Loss:1.4576 Policy:0.4476 Value:1.0100
Game 0 ended in: 1.0
Game 1 ended in: -1.0
Game 2 ended in: -1.0
Game 3 ended in: -1.0
Game 4 ended in: 1.0
Game 5 ended in: 1.0
Game 6 ended in: 1.0
Game 7 ended in: 1.0
Game 8 ended in: -1.0
Game 9 ended in: 1.0
Game 10 ended in: -1.0
Game 11 ended in: 1.0


In [ ]:
torch.save(alphazero.model.state_dict(), "alphazeromdodel.pt")

In [ ]:

game = ConnectFour()

while not game.is_terminal:
    print(game)
    action = int(input("Enter action: "))
    game.play_action(action)

    model_action = alphazero.MCTS(game, train=False)
    game.play_action(model_action)

print(game)


[[0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]]
Current Player: 1
Terminal: False
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [-1  0  0  0  0  1  0]]
Current Player: 1
Terminal: False
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  1  0]
 [-1  0 -1  0  0  1  0]]
Current Player: 1
Terminal: False
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0 -1  0]
 [ 0  0  0  0  0  1  0]
 [ 0  0  0  0  0  1  0]
 [-1  0 -1  0  0  1  0]]
Current Player: 1
Terminal: False
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0 -1  0]
 [ 0  0  0  0  0  1  0]
 [ 0  0  0  0  0  1  0]
 [-1 -1 -1  0  0  1  1]]
Current Player: 1
Terminal: False
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0 -1  0]
 [ 0  0  0  0  0  1  0]
 [ 0  0  0  0  0  1  1]
 [-1 -1 -1 -1  0  1  1]]
Cu